# Banking77 Reasoning-Distilled SLM (Phase 3: QLoRA Fine-Tuning)

This notebook fine-tunes **Qwen-2.5-3B-Instruct** on a free Kaggle GPU (T4 / P100) using 4-bit QLoRA on the distilled Banking77 dataset.

### Training Optimizations & Techniques:
- **Base Model**: `Qwen/Qwen2.5-3B-Instruct` (State-of-the-art 3B reasoning model with open weights, no gating).
- **4-bit NF4 Quantization (BitsAndBytes)**: Drops base model VRAM usage from ~24 GB to ~1.9 GB; peak training VRAM is ~5.5 GB (fits easily inside Kaggle's 16 GB VRAM).
- **LoRA Adaptation**: Injects trainable rank-16 adapters on all projection matrices (`q, k, v, o, gate, up, down`).
- **Stable Float32 Adapters with Float16 Compute**: Eliminates PyTorch AMP GradScaler unscale bugs on Tesla T4 while computing GEMM in Tensor Core Float16.
- **Adaptive Learning Rate**: Cosine decay with 5% warmup (`lr=2e-4`).
- **Step-wise Evaluation**: Evaluates every 100 steps on `val.jsonl`.
- **Early Stopping**: Halts training if `eval_loss` stalls for 3 consecutive evaluations (patience=3).
- **Checkpoint Saving & Resume**: Saves checkpoints every 100 steps (keeps last 3), permitting resumption across Kaggle sessions.
- **Effective Batch Size**: 16 (per-device batch size 4 x gradient accumulation steps 4).


In [ ]:
# 1. Install required packages on Kaggle
!pip install -q "transformers>=4.45.0" "peft>=0.12.0" "trl>=0.11.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" datasets matplotlib


In [ ]:
# 2. Check GPU environment
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU accelerator not detected. Please enable GPU in Kaggle Notebook settings (Settings -> Accelerator -> GPU T4 x 2).")


In [ ]:
# 3. Locate Dataset Paths (Auto-discovers in /kaggle/input or local directories)
import os
from pathlib import Path

train_path = None
val_path = None

# Search /kaggle/input recursively
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    for root, dirs, files in os.walk(kaggle_input):
        if "train.jsonl" in files:
            train_path = Path(root) / "train.jsonl"
        if "val.jsonl" in files:
            val_path = Path(root) / "val.jsonl"

# Fallback to local paths if running outside Kaggle input
if train_path is None or not train_path.exists():
    local_candidates = [Path("./data/processed"), Path("../data/processed"), Path("./")]
    for d in local_candidates:
        if (d / "train.jsonl").exists():
            train_path = d / "train.jsonl"
            val_path = d / "val.jsonl"
            break

print(f"Using Train Path: {train_path}")
print(f"Using Val Path:   {val_path}")
assert train_path and train_path.exists(), "Could not find train.jsonl! Please attach the dataset."
assert val_path and val_path.exists(), "Could not find val.jsonl! Please attach the dataset."


In [ ]:
# 4. Load Datasets and Tokenizer
import json
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = load_jsonl(train_path)
val_records = load_jsonl(val_path)
print(f"Loaded {len(train_records)} train records and {len(val_records)} validation records.")

def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = Dataset.from_list(train_records).map(apply_template, desc="Formatting train")
val_ds = Dataset.from_list(val_records).map(apply_template, desc="Formatting val")

print("\nSample formatted prompt snippet:")
print(train_ds[0]["text"][:300] + "...")


In [ ]:
# 5. Load Model with 4-bit NormalFloat Quantization & LoRA Configuration
import torch
from transformers import AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.torch_dtype = torch.float16
config.use_cache = False

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

# Ensure trainable adapter parameters are in Float32 for maximum gradient stability
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

print("Model loaded with 4-bit NF4 and Float32 LoRA adapters:")
model.print_trainable_parameters()


In [ ]:
# 6. Configure Training Arguments (Robust dynamic signature matching, FP16-safe)
import inspect
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

OUTPUT_DIR = "/kaggle/working/checkpoints"
FINAL_ADAPTER_DIR = "/kaggle/working/qwen2.5_3b_banking77_final"

# Determine supported arguments for TrainingArguments in current environment
ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

desired_args = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,      # Effective batch size = 16
    "per_device_eval_batch_size": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",         # Adaptive Cosine decay
    "warmup_steps": 38,                    # ~5% warmup across 756 steps
    "weight_decay": 0.01,
    "fp16": False,                         # Avoids GradScaler unscale bugs; 4-bit base computes in Float16 via BitsAndBytes
    "bf16": False,
    "max_grad_norm": 1.0,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 100,
    "save_total_limit": 3,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "gradient_checkpointing": True,
    "seed": 77,
    "report_to": "none",
}

# Handle eval_strategy vs evaluation_strategy
if "eval_strategy" in ta_params:
    desired_args["eval_strategy"] = "steps"
    desired_args["save_strategy"] = "steps"
elif "evaluation_strategy" in ta_params:
    desired_args["evaluation_strategy"] = "steps"
    desired_args["save_strategy"] = "steps"

# Filter only arguments supported by this version of transformers
filtered_ta_args = {k: v for k, v in desired_args.items() if k in ta_params}
print(f"Initializing TrainingArguments with {len(filtered_ta_args)} validated parameters...")
training_args = TrainingArguments(**filtered_ta_args)

# Early stopping halts if eval_loss stalls for 3 consecutive checks (300 steps)
early_stopping = EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.005)

# Dynamically inspect SFTTrainer signature
sft_sig = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
sft_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "callbacks": [early_stopping],
}
if "dataset_text_field" in sft_sig:
    sft_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_sig:
    sft_kwargs["max_seq_length"] = 640
if "processing_class" in sft_sig:
    sft_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_sig:
    sft_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**sft_kwargs)
print("SFTTrainer successfully initialized!")


In [ ]:
# 7. Train Model (with Auto-Resume Support across sessions)
import os
from pathlib import Path

resume_ckpt = None
if Path(OUTPUT_DIR).exists():
    ckpts = [p for p in Path(OUTPUT_DIR).glob("checkpoint-*") if (p / "trainer_state.json").exists()]
    if ckpts:
        latest = max(ckpts, key=lambda p: int(p.name.split("-")[1]))
        print(f"Detected existing checkpoint: {latest}. Resuming from here...")
        resume_ckpt = str(latest)

print("\n--- Starting SFT Training ---")
train_result = trainer.train(resume_from_checkpoint=resume_ckpt)

# Save best adapter and tokenizer
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"\nSuccessfully saved best model adapter to {FINAL_ADAPTER_DIR}!")


In [ ]:
# 8. Plot Loss Curves (Train vs Evaluation Loss)
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps, train_losses = [], []
eval_steps, eval_losses = [], []

for log in history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="Train Loss (SFT)", color="#1f77b4", alpha=0.8)
if eval_losses:
    plt.plot(eval_steps, eval_losses, label="Validation Loss", color="#ff7f0e", marker="o", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Cross Entropy Loss")
plt.title("Qwen-2.5-3B QLoRA Training on Banking77 CoT")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("/kaggle/working/loss_curves.png", dpi=150)
plt.show()


In [ ]:
# 9. Test Inference (Testing Think & Ambiguity on Kaggle)
system_prompt = train_records[0]["messages"][0]["content"]
test_queries = [
    "Why was my payment declined at the supermarket?",
    "The ATM swallowed my card and won't give it back.",
    "I have a payment problem",  # Ambiguous query
]

model.eval()
for q in test_queries:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n========================================")
    print(f"User Query: {q}")
    print(f"Model Response:\n{response}")
